# 2. Chunking Strategies

**RAG Pipeline Series — Notebook 2**

Chunking is where a lot of RAG quality gets won or lost. In notebook 1 we turned `rag.pdf` into 58 page-level `Document` objects. But a "page" is an arbitrary boundary — picked by a printer, not by meaning. `rag.pdf` (a 15-chapter RAG course) mixes several kinds of content that each want to be chunked differently:

- **Running prose** — multi-paragraph explanations inside each chapter section.
- **Numbered chapters and subsections** (`CHAPTER 01`, `1.1 Why Do Large Language Models Need RAG?`) — a natural, citable structure worth preserving as metadata.
- **Bulleted lists** — e.g. every chapter ends with a "Key Takeaways" block of several bullets that only make sense together.
- **Tables** — e.g. `Table 4.1: Popular embedding models comparison`, `Table 5.1: Popular vector databases comparison` — rows that are meaningless once split apart. (Chapter 3 of this very PDF even has a `Table 3.1: Chunking strategy comparison` — so the document is, fittingly, self-aware about this problem.)
- **Callout boxes** — "Real-World Analogy", "Concrete Example", "FEYNMAN TECHNIQUE" sidebars.

In this notebook we build six chunking strategies of increasing sophistication, run each one against the *real* `rag.pdf` text, and look at exactly where each one cuts — including the failure modes on this document's tables and bullet lists.

## Setup

In [ ]:
%pip install -q -U langchain langchain-community langchain-text-splitters langchain-core pypdf pandas tiktoken

## 1. Loading `rag.pdf`

Same loading approach as notebook 1: a Colab upload widget, falling back to the local `../dataset/rag.pdf` path when running outside Colab.

In [ ]:
# Only runs inside Colab. Opens a file picker; select rag.pdf.
# Safe to skip this cell if you're running locally and already have the file on disk.
from rag_utils import maybe_colab_upload

maybe_colab_upload()

In [2]:
from rag_utils import resolve_pdf_path, load_pdf

# Prefer the Colab upload location; fall back to this repo's dataset/ folder
# when running locally (rag-notebooks/ and dataset/ are sibling folders).
PDF_PATH = resolve_pdf_path()

pages = load_pdf(PDF_PATH)  # List[Document], one per PDF page

print(f"Loaded {len(pages)} pages from {PDF_PATH}")

Loaded 58 pages from ..\dataset\rag.pdf


## 2. Stitching pages into one document — and stripping running headers/footers

Every page of `rag.pdf` repeats a running header (`RAG to Agentic RAG — Comprehensive Course`) and a `Page N` footer line. Left in place, those two lines get baked into nearly every chunk we produce. We strip them before chunking — a small but important preprocessing step for any real PDF.

In [3]:
import re

from rag_utils import clean_page

cleaned_pages = [clean_page(doc.page_content) for doc in pages]
full_text = "\n\n".join(cleaned_pages)

print(f"Full document: {len(full_text)} characters (~{len(full_text.split())} words) across {len(pages)} pages")

Full document: 110023 characters (~16109 words) across 58 pages


## 3. A quick look at the document's structure

Before picking a chunking strategy, look at what's actually in the text: chapter headers, numbered subsections, bullet lists, table captions, and callout boxes. This particular PDF extracts its bullet glyph as the control character `\x7f` (a font-encoding quirk of the source file) — a good reminder that real-world PDF extraction always needs a sanity check before you build on top of it.

In [4]:
from rag_utils import CHAPTER_RE as chapter_re

section_re = re.compile(r"^(\d+\.\d+)\s+(.+)$", re.MULTILINE)
bullet_re = re.compile(r"^\x7f\s*(.+)$", re.MULTILINE)
table_re = re.compile(r"^\s*Table (\d+\.\d+):\s*(.+)$", re.MULTILINE)
callout_re = re.compile(r"^(Real-World Analogy|Concrete Example|Key Takeaways|FEYNMAN TECHNIQUE)", re.MULTILINE)

print(f"Chapters found:       {len(chapter_re.findall(full_text))}")
print(f"Subsections found:    {len(section_re.findall(full_text))}")
print(f"Bullet lines found:   {len(bullet_re.findall(full_text))}")
print(f"Callout blocks found: {len(callout_re.findall(full_text))}")
print("Tables found:")
for num, caption in table_re.findall(full_text):
    print(f"  Table {num}: {caption}")

Chapters found:       15
Subsections found:    74
Bullet lines found:   316
Callout blocks found: 60
Tables found:
  Table 1.1: Comparing knowledge access strategies for LLMs
  Table 2.1: Key milestones in information retrieval
  Table 3.1: Chunking strategy comparison
  Table 4.1: Popular embedding models comparison
  Table 5.1: Popular vector databases comparison
  Table 7.1: Retrieval evaluation metrics summary
  Table 8.1: Bi-encoder vs Cross-encoder comparison
  Table 10.1: Generation parameters and RAG recommendations
  Table 11.1: Generation evaluation metrics summary
  Table 12.1: RAG vs Fine-tuning comparison
  Table 13.1: Tool types for retrieval in RAG systems
  Table 15.1: Cost optimization strategies for production RAG


A single table also shows why table extraction is messy: `PyPDFLoader` (like most PDF-to-text tools) reads left-to-right, top-to-bottom, which flattens a table's rows and columns into one linear stream of cell values with no delimiters. Here's the raw text of the vector-database comparison table (`Table 5.1`) — vendor names, deployment types, and descriptions all run together with no structure marking where one row ends and the next begins.

In [5]:
table_start = full_text.find("Pinecone")
print(full_text[table_start:table_start + 500])

Pinecone
Managed Cloud
Zero-ops, scalable, serverless option
Startups, rapid deployment
Weaviate
Open-source /
Cloud
GraphQL API, multi-modal, modules
ecosystem
Multi-modal RAG
Chroma
Open-source
Simple API, embeddable, great for
development
Development, small
projects
Milvus
Open-source /
Cloud
Massive scale (billion vectors),
distributed
Enterprise, huge scale
pgvector
PostgreSQL ext.
Reuses existing Postgres infra, simple
ops
Existing Postgres shops
FAISS (Meta)
Library
State-of-the-art algor


## Strategy 1 — `CharacterTextSplitter` (naive fixed-length)

The simplest possible splitter: cut every `chunk_size` characters with `separator=""`, ignoring sentence, paragraph, or table structure entirely. Included as a baseline / cautionary example, not a recommendation.

In [6]:
from langchain_text_splitters import CharacterTextSplitter

naive_splitter = CharacterTextSplitter(separator="", chunk_size=500, chunk_overlap=0)
naive_chunks = naive_splitter.split_text(full_text)

print(f"{len(naive_chunks)} chunks\n")
for i, chunk in enumerate(naive_chunks[:3]):
    print(f"--- chunk {i} (ends mid-word/mid-table-row) ---")
    print(f"...{chunk[-120:]}")
    print()

221 chunks

--- chunk 0 (ends mid-word/mid-table-row) ---
...atabases & Indexing
Ch 13 RAG with Tools
Ch 06 Retrieval Techniques
Ch 14 Agentic RAG
Ch 07 Retrieval Evaluation
Ch 15 P

--- chunk 1 (ends mid-word/mid-table-row) ---
...l reasoning behind RAG — why it exists, what problems it solves, and
how it works at a high level.
1.1 Why Do Large Lang

--- chunk 2 (ends mid-word/mid-table-row) ---
...ines. They learn statistical relationships
between tokens, phrases, and concepts. When you ask an LLM a factual question



Notice how often a chunk boundary lands mid-word or mid-table-row (e.g. splitting a vendor name from its description in the table above). This is the strategy to avoid whenever the source document has any structure worth preserving.

## Strategy 2 — `RecursiveCharacterTextSplitter` (paragraph/sentence aware)

Tries a list of separators in order — `["\n\n", "\n", ". ", " ", ""]` — falling back to a harder cut only when a chunk still doesn't fit. This is the default splitter to reach for. We'll use `chunk_size=800, chunk_overlap=100` as this series' baseline (chapter subsections here typically run 500-2000+ characters).

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE, CHUNK_OVERLAP = 800, 100

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
recursive_chunks = recursive_splitter.split_text(full_text)

print(f"{len(recursive_chunks)} chunks\n")
for i, chunk in enumerate(recursive_chunks[:3]):
    print(f"--- chunk {i} (ends on a cleaner boundary) ---")
    print(f"...{chunk[-120:]}")
    print()

182 chunks

--- chunk 0 (ends on a cleaner boundary) ---
...Ch 06 Retrieval Techniques
Ch 14 Agentic RAG
Ch 07 Retrieval Evaluation
Ch 15 Production Best Practices
Ch 08 Re-ranking

--- chunk 1 (ends on a cleaner boundary) ---
...eir neural network
weights — a form of parametric memory. This is impressive, but it comes with fundamental
limitations.

--- chunk 2 (ends on a cleaner boundary) ---
...ontext window before
generating an answer. This makes the model's output grounded in verifiable, up-to-date information.



Compare these chunk endings to Strategy 1 — far more land on sentence or clause boundaries. It still knows nothing about chapters, bullet lists, or tables, though: a "Key Takeaways" list or a comparison table can still be split arbitrarily wherever the character count happens to run out.

## Strategy 3 — Token-based chunking

Character count is only a proxy for what an embedding or LLM model actually consumes: tokens. `RecursiveCharacterTextSplitter.from_tiktoken_encoder` measures `chunk_size` in tokens instead — useful once you're sizing chunks against a specific embedding model's max sequence length (e.g. 512 tokens for many sentence-transformer models, 8191 for `text-embedding-3-*`).

In [8]:
import tiktoken

token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=200, chunk_overlap=30
)
token_chunks = token_splitter.split_text(full_text)

enc = tiktoken.get_encoding("cl100k_base")
print(f"{len(token_chunks)} chunks")
for i, chunk in enumerate(token_chunks[:3]):
    print(f"chunk {i}: {len(enc.encode(chunk))} tokens, {len(chunk)} characters")

164 chunks
chunk 0: 154 tokens, 541 characters
chunk 1: 181 tokens, 879 characters
chunk 2: 179 tokens, 903 characters


## Strategy 4 — Chapter/section-aware chunking

`rag.pdf` has real structure: 15 numbered chapters (`CHAPTER 01` … `CHAPTER 15`), each split into numbered subsections (`1.1`, `1.2`, …). We can parse that structure and attach it as metadata on every chunk, so retrieval results can be filtered or displayed as "Chapter 4, Vector Databases & Indexing" instead of an anonymous blob of text. We also make sure no chunk ever spans two chapters, by running `RecursiveCharacterTextSplitter` *within* each chapter rather than across the whole document.

In [9]:
from rag_utils import split_into_chapters

chapters = split_into_chapters(full_text)
print(f"{len(chapters)} chapters parsed\n")
for c in chapters[:3]:
    print(f"Ch {c['chapter_num']}: {c['chapter_title']} ({len(c['text'])} chars)")

15 chapters parsed

Ch 01: Introduction to RAG (8635 chars)
Ch 02: Evolution of Retrieval (7670 chars)
Ch 03: Data Ingestion (8048 chars)


In [10]:
from rag_utils import chunk_chapters

chapter_aware_docs = chunk_chapters(chapters, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print(f"{len(chapter_aware_docs)} chunks, each tagged with chapter metadata\n")
print(chapter_aware_docs[10].metadata)
print(chapter_aware_docs[10].page_content[:200])

181 chunks, each tagged with chapter metadata

{'chapter_num': '01', 'chapter_title': 'Introduction to RAG'}
 Step 1 — Query Reception: A user submits a natural-language query.
 Step 2 — Query Embedding: The query is embedded into the same vector space as the
documents.
 Step 3 — Retrieval: The vector dat


Every chunk now carries `chapter_num` / `chapter_title` metadata for free — useful for citation, filtering (e.g. "only search Chapter 5 — Vector Databases & Indexing"), and debugging retrieval results. This costs nothing extra at query time, since it's computed once, during indexing.

## Strategy 5 — Bullet-list-aware chunking

Several sections are bulleted lists where each item only makes sense as part of the whole — most notably the "Key Takeaways" block that ends every chapter (several `\x7f`-marked bullets). Splitting a takeaways block into two chunks because item 3 crossed a character boundary loses the "these are the key points of this chapter" framing. We detect contiguous bullet runs (including a heading like "Key Takeaways" directly above them, and wrapped lines within a single bullet) and keep each run as one chunk, only falling back to `RecursiveCharacterTextSplitter` if a run is unusually long.

In [11]:
HEADING_RE = re.compile(r"^(Key Takeaways|Real-World Analogy|Concrete Example|FEYNMAN TECHNIQUE)")
STRUCTURAL_RE = re.compile(r"^(CHAPTER \d+|\d+\.\d+\s)")

def chunk_preserving_bullet_runs(text: str, chunk_size=800, chunk_overlap=100):
    lines = text.split("\n")
    blocks, buffer = [], []
    in_bullet_run, just_saw_heading = False, False

    def flush():
        if buffer:
            blocks.append("\n".join(buffer).strip())
            buffer.clear()

    for line in lines:
        stripped = line.strip()
        is_bullet = stripped.startswith("\x7f")
        is_heading = bool(HEADING_RE.match(stripped))
        is_structural = bool(STRUCTURAL_RE.match(stripped))
        is_blank = stripped == ""

        if is_heading:
            flush()  # start a new block for the heading (e.g. "Key Takeaways")
            in_bullet_run = False
        elif is_bullet and not in_bullet_run and not just_saw_heading:
            flush()  # start of a bare bullet run with no heading above it
        elif (is_blank or is_structural) and in_bullet_run:
            flush()  # blank line / new chapter or section closes the run
            in_bullet_run = False

        buffer.append(line)  # non-bullet lines here are wrapped continuations of the last bullet
        if is_bullet:
            in_bullet_run = True
        just_saw_heading = is_heading
    flush()

    fallback = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = []
    for block in blocks:
        if len(block) <= chunk_size:
            chunks.append(block)
        else:
            chunks.extend(fallback.split_text(block))
    return [c for c in chunks if c.strip()]

bullet_aware_chunks = chunk_preserving_bullet_runs(full_text, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
print(f"{len(bullet_aware_chunks)} chunks\n")

# Find a "Key Takeaways" chunk and confirm every bullet (and wrapped line) stayed together
takeaway_chunk = next(c for c in bullet_aware_chunks if "Key Takeaways" in c)
print(takeaway_chunk)

237 chunks

Key Takeaways — Chapter 1
 LLMs rely on parametric memory that is static, finite, and prone to hallucinations.
 Context window constraints make naive document stuffing impractical at scale.
 RAG combines the language understanding of LLMs with dynamic, updatable external knowledge.
 The RAG pipeline has two phases: an offline indexing phase and an online inference phase.
 RAG is not a replacement for LLMs but a powerful architectural pattern that makes them more
reliable.


## Strategy 6 — Table-aware chunking

Tables are the highest-risk content for naive chunking: a comparison table split mid-row (e.g. separating `Pinecone` from `Managed Cloud, Zero-ops...`) becomes gibberish once embedded. We locate every `Table N.N: caption` in the document, walk backward to the nearest paragraph break before it to capture the table's body, and emit the whole table as one atomic chunk — never split further, regardless of size.

Note: `PyPDFLoader` extracts tables as an unstructured stream of cell text with no row/column delimiters (see the raw text in section 3 above) — good enough to keep a table's content together, but not to reconstruct exact rows and columns. For pipelines that need real row/column structure, swap in a table-aware extractor like `pdfplumber`'s `extract_tables()`, `camelot`, or the `unstructured` library's table partitioning instead of `PyPDFLoader`.

In [12]:
def chunk_preserving_tables(text: str, chunk_size=800, chunk_overlap=100):
    table_matches = list(table_re.finditer(text))
    table_spans = []
    prev_end = 0
    for m in table_matches:
        body_start = text.rfind("\n\n", prev_end, m.start())
        body_start = body_start + 2 if body_start != -1 else prev_end
        table_spans.append((body_start, m.end()))
        prev_end = m.end()

    fallback = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks, cursor = [], 0
    for start, end in table_spans:
        if start > cursor:
            chunks.extend(fallback.split_text(text[cursor:start]))
        chunks.append(text[start:end].strip())  # whole table, kept atomic
        cursor = end
    if cursor < len(text):
        chunks.extend(fallback.split_text(text[cursor:]))
    return [c for c in chunks if c.strip()]

table_aware_chunks = chunk_preserving_tables(full_text, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
print(f"{len(table_aware_chunks)} chunks\n")

# Show the "Chunking strategy comparison" table (Table 3.1) kept fully intact
table_chunk = next(c for c in table_aware_chunks if "Chunking strategy comparison" in c)
print(table_chunk)

175 chunks

Strategy
 Chunk
Quality
 Semantic
Coherence
 Complexit
 y
 Best For
Fixed-size
Low-Medium
Low
Very Low
Quick prototypes
Recursive character
Medium
Medium
Low
General-purpose
Sentence-based
Medium-High
High
Medium
Narrative documents
Semantic chunking
High
Very High
High
Quality-critical systems
Structure-aware
High
High
Medium-Hig
h
Structured documents (PDFs,
HTML)
Parent document
High
High
High
Long documents with
hierarchy
 Table 3.1: Chunking strategy comparison


## Comparing all six strategies

A quick side-by-side on chunk count and size distribution. Lower chunk count isn't inherently "better" — the right size depends on what downstream retrieval needs — but a wildly different size spread points to a splitter that isn't respecting document structure.

In [13]:
import pandas as pd

strategies = {
    "1. Naive fixed-length": naive_chunks,
    "2. RecursiveCharacterTextSplitter": recursive_chunks,
    "3. Token-based (tiktoken)": token_chunks,
    "4. Chapter-aware": [d.page_content for d in chapter_aware_docs],
    "5. Bullet-list-aware": bullet_aware_chunks,
    "6. Table-aware": table_aware_chunks,
}

summary = pd.DataFrame([
    {
        "strategy": name,
        "num_chunks": len(chunks),
        "avg_chars": round(sum(len(c) for c in chunks) / len(chunks), 1),
        "min_chars": min(len(c) for c in chunks),
        "max_chars": max(len(c) for c in chunks),
    }
    for name, chunks in strategies.items()
])
summary

,strategy,num_chunks,avg_chars,min_chars,max_chars
0,1. Naive fixed-length,221,497.5,23,500
1,2. RecursiveCharacterTextSplitter,182,654.9,107,799
2,3. Token-based (tiktoken),164,733.1,94,1016
3,4. Chapter-aware,181,653.0,107,800
4,5. Bullet-list-aware,237,479.0,25,800
5,6. Table-aware,175,672.0,107,2409


## Takeaways

- The right chunking strategy depends on what's *in* the document, not just its overall length — `rag.pdf` needed different handling for prose, chapter structure, bullet lists, and tables.
- `CharacterTextSplitter` with `separator=""` is a cautionary baseline — it cuts mid-word and mid-table-row with no regard for structure.
- `RecursiveCharacterTextSplitter` (`chunk_size=800, chunk_overlap=100`) is the default to reach for — it's used as the baseline splitter for the rest of this series.
- Token-based chunking matters once chunk size needs to respect an embedding model's max sequence length, not just an arbitrary character count.
- Structure-aware strategies (chapter metadata, bullet-run preservation, atomic tables) cost more code, but directly fix failure modes a generic splitter can't see — a table split mid-row, or a takeaways list broken into disconnected chunks, actively hurts retrieval quality.
- These strategies aren't mutually exclusive. A production pipeline for this document would likely combine #4 (chapter metadata) with #5/#6 (structure preservation), using `RecursiveCharacterTextSplitter` only as the final fallback for oversized prose runs.

**Next up (notebook 3):** turning these chunks into **keyword embeddings** (sparse retrieval).